In [3]:
from rdkit import Chem
from rdkit.Chem import AllChem
from openff.toolkit import ForceField, Molecule, Topology

def load_with_template(sdf, smiles, name):
    '''
    Use coordinates from sdf exported from ligated PDB
    Use topology from SMILES
    Use SMILES topology to fix incorrect topology of sdf
    '''
    pdb_mol = Chem.MolFromMolFile(sdf, removeHs=False, sanitize=False)
    heavy = Chem.RemoveHs(pdb_mol, sanitize=False)          # drop the file's H atoms
    template = Chem.MolFromSmiles(smiles)                   # heavy-atom template, no AddHs
    fixed = AllChem.AssignBondOrdersFromTemplate(template, heavy)
    fixed = Chem.AddHs(fixed, addCoords=True)               # H's to match the SMILES' protonation
    Chem.AssignStereochemistryFrom3D(fixed)
    mol = Molecule.from_rdkit(fixed)
    mol.name = name
    return mol

d = "/n/home00/ewang03/lab/GroEL/openMM_inputs/m20i/"
tmp_plus = load_with_template(d + "tmp.sdf", "COc1cc(Cc2c[nH+]c(N)nc2N)cc(OC)c1OC", "TMP")
Chem.MolToMolFile(tmp_plus.to_rdkit(), d + "tmp_plus_cgenff.sdf")

tmp = load_with_template(d + "tmp.sdf", "COc1cc(Cc2cnc(N)nc2N)cc(OC)c1OC", "TMP")
Chem.MolToMolFile(tmp.to_rdkit(), d + "tmp_cgenff.sdf")

nadph = load_with_template(d + "nadph.sdf", "NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)([O-])OP(=O)([O-])OC[C@H]3O[C@@H](n4cnc5c(N)ncnc54)[C@H](OP(=O)([O-])[O-])[C@@H]3O)[C@@H](O)[C@H]2O)C=CC1", "NADPH")
Chem.MolToMolFile(nadph.to_rdkit(), d + "nadph_cgenff.sdf")


[21:54:22] WARNING: More than one matching pattern found - picking one

[21:54:22] WARNING: More than one matching pattern found - picking one

[21:54:23] WARNING: More than one matching pattern found - picking one



In [4]:
def fix_mol2(path, resname):
    '''give every atom a unique name (element + count, e.g. C1, N2, H15) and set the molecule/residue name'''
    lines = open(path).read().splitlines()
    out, section, counts = [], None, {}
    for l in lines:
        if l.startswith("@<TRIPOS>"):
            section = l
            out.append(l)
            continue
        if section == "@<TRIPOS>MOLECULE" and out[-1] == "@<TRIPOS>MOLECULE":
            l = resname                                   # molecule name -> CGenFF RESI name
        elif section == "@<TRIPOS>ATOM" and l.strip():
            f = l.split()
            el = f[5].split(".")[0]
            counts[el] = counts.get(el, 0) + 1
            f[1], f[7] = f"{el}{counts[el]}", resname
            l = (f"{int(f[0]):7d} {f[1]:<5s}{float(f[2]):10.4f}{float(f[3]):10.4f}{float(f[4]):10.4f} "
                 f"{f[5]:<6s}{int(f[6]):>3d}  {f[7]:<6s}{float(f[8]):9.4f}")
        out.append(l)
    open(path, "w").write("\n".join(out) + "\n")

d = "/n/home00/ewang03/lab/GroEL/openMM_inputs/"
for f, res in [("tmp_plus.mol2", "TMP"), ("tmp.mol2", "TMP"), ("nadph.mol2", "NDP")]:
    fix_mol2(d + f, res)
